# LSTM — Scheme 1 (Variable Test) — LUMED

> Run on **Google Colab**. Mount your Google Drive and adjust `folder_path` before executing.


In [ ]:
import os, numpy as np, pandas as pd
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ── Define training size (change per experiment step) ──
train_size = 0.95  # 95% training; remainder becomes test set

folder_path = '/content/drive/My Drive/EEG Datasets/LUMED/LUMED CSV/'
csv_files = [f"wavelet_denoised_s{i:02}.csv" for i in range(1, 14)]

data = pd.DataFrame()
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    temp_data = pd.read_csv(file_path)
    temp_data = temp_data.sample(frac=0.8, random_state=42).reset_index(drop=True)
    data = pd.concat([data, temp_data], ignore_index=True)

X = data.drop(columns=['label']).values
try:
    y = data['label'].astype(int).values
except ValueError:
    y = data['label'].apply(lambda x: int(x.strip("[]"))).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Train/Test split with stratification ──
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, train_size=train_size, random_state=42, stratify=y)

# ────────────────────────────────────────────────────────────
# LSTM — Scheme 1 (Variable Test) — LUMED
# ────────────────────────────────────────────────────────────
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense

X_train_in = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_in  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)
X_cv = X_train_in

def build_model():
    m = Sequential([
        Input(shape=(X_train_in.shape[1], 1)),
        LSTM(64, return_sequences=True), Dropout(0.2),
        LSTM(32), Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

num_classes = len(np.unique(y_train))

# ── 5-Fold Cross-Validation ──
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_accuracies = []
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_cv), 1):
    m = build_model()
    m.fit(X_cv[tr_idx], y_train[tr_idx], epochs=10, batch_size=32, verbose=0)
    _, acc = m.evaluate(X_cv[val_idx], y_train[val_idx], verbose=0)
    cv_accuracies.append(acc)
    print(f"Fold {fold} Accuracy: {acc:.4f}")
print(f"5-Fold CV Accuracy with {int(train_size * 100)}% training data: {np.mean(cv_accuracies):.4f} ± {np.std(cv_accuracies):.4f} (SD)")

final_model = build_model()
final_model.fit(X_train_in, y_train, epochs=10, batch_size=32, verbose=1,
                validation_data=(X_test_in, y_test))
_, accuracy = final_model.evaluate(X_test_in, y_test, verbose=0)
print(f"Held-out Test Accuracy with {int(train_size * 100)}% training data: {accuracy:.4f}")

y_pred = np.argmax(final_model.predict(X_test_in), axis=1)
cm = confusion_matrix(y_test, y_pred, normalize='true')
plt.figure(figsize=(5, 4))
sns.heatmap(cm * 100, annot=True, fmt=".2f", cmap="Blues")
plt.title(f"Confusion Matrix ({int(train_size * 100)}% Training Data)")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()